# EEGFormer Model Comparison

This notebook rebuilds the publication-style model comparison table from the existing result folders and keeps EEGFormer as the proposed target model.

It uses the same protocol as the reference table: 5 fixed subject-wise folds, repeated over 10 random seeds, for 50 fold-level evaluations per model.

Notes:
- Baseline models are loaded from `all_fold_results.csv` when available.
- EEGConformer is loaded from `results/hypothesis tests/eegconformer_10seeds/seed_*/trial_predictions.csv` and collapsed to the same fold-level evaluation basis.
- EEGConformer is cited as the 2022 IEEE TNSRE paper by Song et al.
- For the published Braindecode-style default configuration on 22 channels and 1000 time samples, EEGConformer has 789,506 trainable parameters.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score

BASE_DIR = Path('../results/hypothesis tests')
OUT_DIR = BASE_DIR / 'eegformer_model_comparison_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_MODEL = 'EEGFormer'

MODELS = [
    {'key': 'ShallowConvNet', 'display': 'ShallowConvNet 2017', 'year': 2017, 'params': '36,522', 'summary_path': BASE_DIR / 'shallowconvnet_10seeds' / 'all_fold_results.csv'},
    {'key': 'EEGNet', 'display': 'EEGNet 2017', 'year': 2017, 'params': '1,746', 'summary_path': BASE_DIR / 'eegnet_10seeds' / 'all_fold_results.csv'},
    {'key': 'CNN-LSTM-EEGNet', 'display': 'CNN-LSTM 2022', 'year': 2022, 'params': '9,052', 'summary_path': BASE_DIR / 'cnn_lstm_eegnet_10seeds' / 'all_fold_results.csv'},
    {'key': 'MultiStream', 'display': 'Multi-Stream 2023', 'year': 2023, 'params': '574,082', 'summary_path': BASE_DIR / 'multistream_10seeds' / 'all_fold_results.csv'},
    {'key': 'IMCBGT', 'display': 'IM-CBGT 2024', 'year': 2024, 'params': '1,195,266', 'summary_path': BASE_DIR / 'imcbgt_10seeds' / 'all_fold_results.csv'},
    {'key': 'TGARNet', 'display': 'T-GARNet 2025', 'year': 2025, 'params': '9,071', 'summary_path': BASE_DIR / 'tgarnet_10seeds' / 'all_fold_results.csv'},
    {'key': 'EEGFormer', 'display': 'EEG-Former 2026 (This work)', 'year': 2026, 'params': '7,322', 'summary_path': BASE_DIR / 'eegformer_10seeds' / 'all_fold_results.csv'},
    {'key': 'EEGConformer', 'display': 'EEGConformer 2022', 'year': 2022, 'params': '789,506', 'trial_path': BASE_DIR / 'eegconformer_10seeds', 'folder_mode': True},
]


def format_mean_std(mean_val, std_val):
    if pd.isna(mean_val) or pd.isna(std_val):
        return 'N/A'
    return f'{mean_val:.1f} +/- {std_val:.1f}'



def _fold_metrics_from_trial_predictions(trials: pd.DataFrame) -> pd.DataFrame:
    required = {'seed', 'fold', 'subject', 'true_label', 'subject_majority_pred'}
    missing = required - set(trials.columns)
    if missing:
        raise ValueError(f"trial_predictions.csv is missing required columns: {sorted(missing)}")

    keep_cols = ['seed', 'fold', 'subject', 'true_label', 'subject_majority_pred']
    if 'subject_mean_prob' in trials.columns:
        keep_cols.append('subject_mean_prob')
    trials = trials[keep_cols].drop_duplicates().copy()
    trials = trials.dropna(subset=['subject'])
    trials['seed'] = pd.to_numeric(trials['seed'], errors='coerce').astype(int)
    trials['fold'] = pd.to_numeric(trials['fold'], errors='coerce').astype(int)
    trials['subject'] = trials['subject'].astype(str)
    trials['true_label'] = pd.to_numeric(trials['true_label'], errors='coerce').astype(int)
    trials['subject_majority_pred'] = pd.to_numeric(trials['subject_majority_pred'], errors='coerce').astype(int)
    if 'subject_mean_prob' in trials.columns:
        trials['subject_mean_prob'] = pd.to_numeric(trials['subject_mean_prob'], errors='coerce').astype(float)

    rows = []
    for (seed, fold), group in trials.groupby(['seed', 'fold'], sort=True):
        y_true = group['true_label'].to_numpy(dtype=int)
        y_pred = group['subject_majority_pred'].to_numpy(dtype=int)
        if 'subject_mean_prob' in group.columns:
            y_score = group['subject_mean_prob'].to_numpy(dtype=float)
        else:
            y_score = y_pred.astype(float)
        if np.unique(y_true).size < 2:
            auc = np.nan
        else:
            auc = float(roc_auc_score(y_true, y_score))
        rows.append({
            'seed': int(seed),
            'fold': int(fold),
            'accuracy': float(accuracy_score(y_true, y_pred)),
            'balanced_acc': float(balanced_accuracy_score(y_true, y_pred)),
            'f1': float(f1_score(y_true, y_pred, zero_division=0)),
            'auc': auc,
        })
    return pd.DataFrame(rows)



def load_fold_metrics(meta):
    summary_path = meta.get('summary_path')
    if summary_path is not None and Path(summary_path).exists():
        metrics = pd.read_csv(summary_path)
        required = {'seed', 'fold', 'accuracy', 'balanced_acc', 'f1', 'auc'}
        missing = required - set(metrics.columns)
        if missing:
            raise ValueError(f"{meta['display']} is missing required columns: {sorted(missing)}")
        metrics = metrics[['seed', 'fold', 'accuracy', 'balanced_acc', 'f1', 'auc']].drop_duplicates().copy()
        metrics['seed'] = pd.to_numeric(metrics['seed'], errors='coerce').astype(int)
        metrics['fold'] = pd.to_numeric(metrics['fold'], errors='coerce').astype(int)
        for column in ['accuracy', 'balanced_acc', 'f1', 'auc']:
            metrics[column] = pd.to_numeric(metrics[column], errors='coerce').astype(float)
        return metrics

    if meta.get('folder_mode'):
        frames = []
        for csv_path in sorted(meta['trial_path'].glob('seed_*/trial_predictions.csv')):
            frames.append(_fold_metrics_from_trial_predictions(pd.read_csv(csv_path)))
        if not frames:
            raise FileNotFoundError(f"No trial_predictions.csv files found under {meta['trial_path']}")
        return pd.concat(frames, ignore_index=True)

    raise FileNotFoundError(f"Missing fold-summary file: {summary_path}")

In [2]:
metric_frames = {}
summary_rows = []

for meta in MODELS:
    metrics = load_fold_metrics(meta)
    metric_frames[meta['key']] = metrics

    summary_rows.append({
        'Model': meta['display'],
        'Year': meta['year'],
        'Parameters': meta['params'],
        'Accuracy (%)': format_mean_std(100.0 * metrics['accuracy'].mean(), 100.0 * metrics['accuracy'].std(ddof=1)),
        'Balanced Acc (%)': format_mean_std(100.0 * metrics['balanced_acc'].mean(), 100.0 * metrics['balanced_acc'].std(ddof=1)),
        'F1 (%)': format_mean_std(100.0 * metrics['f1'].mean(), 100.0 * metrics['f1'].std(ddof=1)),
        'AUC (%)': format_mean_std(100.0 * metrics['auc'].mean(), 100.0 * metrics['auc'].std(ddof=1)),
    })

summary_table = pd.DataFrame(summary_rows)

all_metrics = pd.concat(
    [metrics.assign(model=meta['key']) for meta, metrics in zip(MODELS, metric_frames.values())],
    ignore_index=True,
)
rank_table = all_metrics.pivot_table(index=['seed', 'fold'], columns='model', values='accuracy', aggfunc='first')
rank_table = rank_table.dropna(axis=0, how='any')
avg_ranks = rank_table.rank(axis=1, ascending=False, method='average').mean(axis=0)
summary_table['Average rank'] = summary_table['Model'].map({meta['display']: float(avg_ranks[meta['key']]) for meta in MODELS})
summary_table = summary_table.sort_values('Average rank').reset_index(drop=True)

display(summary_table)

target_display = next(meta['display'] for meta in MODELS if meta['key'] == TARGET_MODEL)
target_accuracy = metric_frames[TARGET_MODEL][['seed', 'fold', 'accuracy']].rename(columns={'accuracy': 'target_accuracy'})

mcnemar_df = pd.DataFrame(
    [{'Model': 'All models', 'Note': 'Subject-level McNemar is not available from fold-summary inputs.'}]
)
odds_ratio_map = {meta['display']: '—' for meta in MODELS}
wilcoxon_map = {meta['display']: '—' for meta in MODELS}
wilcoxon_rows = []

for meta in MODELS:
    if meta['key'] == TARGET_MODEL:
        continue

    compare_accuracy = metric_frames[meta['key']][['seed', 'fold', 'accuracy']].rename(columns={'accuracy': 'compare_accuracy'})
    merged_accuracy = target_accuracy.merge(compare_accuracy, on=['seed', 'fold'], how='inner', validate='one_to_one')
    diff = merged_accuracy['target_accuracy'] - merged_accuracy['compare_accuracy']

    if np.allclose(diff.to_numpy(dtype=float), 0.0):
        stat, pval = 0.0, 1.0
    else:
        res = wilcoxon(merged_accuracy['target_accuracy'], merged_accuracy['compare_accuracy'], alternative='greater', zero_method='wilcox', mode='auto')
        stat, pval = float(res.statistic), float(res.pvalue)

    wilcoxon_rows.append({
        'Model': meta['display'],
        'n_pairs': len(merged_accuracy),
        'Mean diff accuracy': float(diff.mean()),
        'Median diff accuracy': float(np.median(diff)),
        'Wilcoxon W': stat,
        'One-sided p-value': pval,
        'Average rank target': float(avg_ranks[TARGET_MODEL]),
        'Average rank compare': float(avg_ranks[meta['key']]),
    })
    wilcoxon_map[meta['display']] = f'{pval:.6f}'

wilcoxon_df = pd.DataFrame(wilcoxon_rows).sort_values('One-sided p-value').reset_index(drop=True)

display(mcnemar_df)
display(wilcoxon_df)

final_table = summary_table.copy()
final_table['McNemar OR vs EEGFormer'] = final_table['Model'].map(odds_ratio_map)
final_table['Wilcoxon p-value vs EEGFormer'] = final_table['Model'].map(wilcoxon_map)
final_table.loc[final_table['Model'] == target_display, 'McNemar OR vs EEGFormer'] = '—'
final_table.loc[final_table['Model'] == target_display, 'Wilcoxon p-value vs EEGFormer'] = '—'

display(final_table)

,Model,Year,Parameters,Accuracy (%),Balanced Acc (%),F1 (%),AUC (%),Average rank
0,EEG-Former 2026 (This work),2026,"7,322",84.2 +/- 5.4,84.5 +/- 5.5,85.5 +/- 4.7,90.5 +/- 5.2,2.61
1,CNN-LSTM 2022,2022,"9,052",84.2 +/- 7.4,84.4 +/- 7.2,85.8 +/- 6.1,92.0 +/- 5.6,2.65
2,EEGNet 2017,2017,"1,746",80.7 +/- 7.5,81.5 +/- 6.8,82.0 +/- 6.9,89.2 +/- 4.9,3.85
3,ShallowConvNet 2017,2017,"36,522",78.7 +/- 9.4,79.5 +/- 8.6,78.4 +/- 15.4,90.3 +/- 3.6,4.12
4,EEGConformer 2022,2022,"789,506",77.2 +/- 8.2,77.6 +/- 8.4,79.9 +/- 7.0,86.5 +/- 7.2,4.78
5,T-GARNet 2025,2025,"9,071",74.7 +/- 8.3,74.8 +/- 7.0,79.4 +/- 5.6,78.6 +/- 10.0,5.29
6,IM-CBGT 2024,2024,"1,195,266",70.6 +/- 9.6,70.8 +/- 9.6,73.3 +/- 8.2,78.9 +/- 5.0,6.10
7,Multi-Stream 2023,2023,"574,082",69.3 +/- 7.9,69.7 +/- 7.7,72.1 +/- 7.4,76.3 +/- 7.1,6.60


,Model,Note
0,All models,Subject-level McNemar is not available from fo...


,Model,n_pairs,Mean diff accuracy,Median diff accuracy,Wilcoxon W,One-sided p-value,Average rank target,Average rank compare
0,Multi-Stream 2023,50,0.149167,0.166667,1081.0,1.666401e-09,2.61,6.60
1,IM-CBGT 2024,50,0.135833,0.125000,1064.0,5.031659e-09,2.61,6.10
2,T-GARNet 2025,50,0.094167,0.083333,984.0,6.098618e-08,2.61,5.29
3,EEGConformer 2022,50,0.069167,0.062500,862.0,1.225688e-07,2.61,4.78
4,EEGNet 2017,50,0.035000,0.041667,706.5,2.803154e-05,2.61,3.85
5,ShallowConvNet 2017,50,0.055000,0.041667,632.0,3.498778e-04,2.61,4.12
6,CNN-LSTM 2022,50,-0.000833,0.000000,346.0,5.333710e-01,2.61,2.65


,Model,Year,Parameters,Accuracy (%),Balanced Acc (%),F1 (%),AUC (%),Average rank,McNemar OR vs EEGFormer,Wilcoxon p-value vs EEGFormer
0,EEG-Former 2026 (This work),2026,"7,322",84.2 +/- 5.4,84.5 +/- 5.5,85.5 +/- 4.7,90.5 +/- 5.2,2.61,—,—
1,CNN-LSTM 2022,2022,"9,052",84.2 +/- 7.4,84.4 +/- 7.2,85.8 +/- 6.1,92.0 +/- 5.6,2.65,—,0.533371
2,EEGNet 2017,2017,"1,746",80.7 +/- 7.5,81.5 +/- 6.8,82.0 +/- 6.9,89.2 +/- 4.9,3.85,—,0.000028
3,ShallowConvNet 2017,2017,"36,522",78.7 +/- 9.4,79.5 +/- 8.6,78.4 +/- 15.4,90.3 +/- 3.6,4.12,—,0.000350
4,EEGConformer 2022,2022,"789,506",77.2 +/- 8.2,77.6 +/- 8.4,79.9 +/- 7.0,86.5 +/- 7.2,4.78,—,0.000000
5,T-GARNet 2025,2025,"9,071",74.7 +/- 8.3,74.8 +/- 7.0,79.4 +/- 5.6,78.6 +/- 10.0,5.29,—,0.000000
6,IM-CBGT 2024,2024,"1,195,266",70.6 +/- 9.6,70.8 +/- 9.6,73.3 +/- 8.2,78.9 +/- 5.0,6.10,—,0.000000
7,Multi-Stream 2023,2023,"574,082",69.3 +/- 7.9,69.7 +/- 7.7,72.1 +/- 7.4,76.3 +/- 7.1,6.60,—,0.000000


In [3]:
summary_path = OUT_DIR / 'eegformer_summary_table.csv'
mcnemar_path = OUT_DIR / 'eegformer_mcnemar.csv'
wilcoxon_path = OUT_DIR / 'eegformer_wilcoxon.csv'
latex_path = OUT_DIR / 'eegformer_summary_table.tex'

summary_table.to_csv(summary_path, index=False)
mcnemar_df.to_csv(mcnemar_path, index=False)
wilcoxon_df.to_csv(wilcoxon_path, index=False)
latex_path.write_text(final_table.to_latex(index=False, escape=False), encoding='utf-8')

print('Saved files:')
print(summary_path)
print(mcnemar_path)
print(wilcoxon_path)
print(latex_path)

Saved files:
../results/hypothesis tests/eegformer_model_comparison_outputs/eegformer_summary_table.csv
../results/hypothesis tests/eegformer_model_comparison_outputs/eegformer_mcnemar.csv
../results/hypothesis tests/eegformer_model_comparison_outputs/eegformer_wilcoxon.csv
../results/hypothesis tests/eegformer_model_comparison_outputs/eegformer_summary_table.tex
